# GridWise Notebook 2: XGBoost Forecasting Training
Trains three models (demand, solar, surplus), evaluates performance, creates plots, and saves artifacts to Drive.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', 'xgboost', 'scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'joblib', '-q'], check=True)

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

from google.colab import drive
drive.mount('/content/drive')

DATA_PATH = '/content/drive/MyDrive/gridwise_data/karnataka_energy_2019_2024.csv'
MODEL_PATH = '/content/drive/MyDrive/gridwise_data/gridwise_xgb.pkl'

In [ ]:
df = pd.read_csv(DATA_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

encoders = {}
for col in ['district', 'city', 'season', 'zone_type']:
    le = LabelEncoder()
    df[f'{col}_enc'] = le.fit_transform(df[col].astype(str))
    encoders[col] = le

df['lag_1h'] = df.groupby(['district', 'city'])['adjusted_demand_kWh'].shift(1)
df['lag_24h'] = df.groupby(['district', 'city'])['adjusted_demand_kWh'].shift(24)
df['lag_168h'] = df.groupby(['district', 'city'])['adjusted_demand_kWh'].shift(168)

df['rolling_mean_24h'] = (
    df.groupby(['district', 'city'])['adjusted_demand_kWh']
      .rolling(window=24, min_periods=1)
      .mean()
      .reset_index(level=[0, 1], drop=True)
)

df = df.dropna(subset=['lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h']).copy()

feature_cols = [
    'hour', 'day_of_week', 'month', 'year', 'is_weekend', 'is_holiday', 'season_enc',
    'temperature_C', 'humidity_pct', 'solar_irradiance_Wm2', 'wind_speed_kmh',
    'is_rainy', 'is_cloudy', 'peak_flag', 'district_enc', 'city_enc', 'zone_type_enc',
    'prosumer_pct', 'household_count', 'panel_efficiency',
    'lag_1h', 'lag_24h', 'lag_168h', 'rolling_mean_24h',
    'weather_adjustment_factor'
]

targets = {
    'demand': 'adjusted_demand_kWh',
    'solar': 'solar_generation_kWh',
    'surplus': 'grid_surplus_kWh',
}

X = df[feature_cols].copy()
print('Dataset shape after feature engineering:', df.shape)
df.head(3)

In [ ]:
train_mask = df['year'].between(2019, 2022)
val_mask = df['year'] == 2023
test_mask = df['year'] == 2024

X_train, X_val, X_test = X[train_mask], X[val_mask], X[test_mask]

y_train = {name: df.loc[train_mask, col] for name, col in targets.items()}
y_val = {name: df.loc[val_mask, col] for name, col in targets.items()}
y_test = {name: df.loc[test_mask, col] for name, col in targets.items()}

print('Train/Val/Test sizes:', len(X_train), len(X_val), len(X_test))

In [ ]:
params = {
    'n_estimators': 800,
    'max_depth': 7,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42,
    'n_jobs': -1
}

models = {}
for name in targets.keys():
    print(f'Training model_{name}...')
    model = XGBRegressor(**params)
    model.fit(
        X_train,
        y_train[name],
        eval_set=[(X_val, y_val[name])],
        verbose=False
    )
    models[name] = model

print('Training complete')

In [ ]:
def mape(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    eps = 1e-6
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), eps))) * 100

preds = {name: models[name].predict(X_test) for name in targets.keys()}

for name in targets.keys():
    y_true = y_test[name]
    y_pred = preds[name]
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape_score = mape(y_true, y_pred)
    print(f'[{name}] MAE={mae:.4f} RMSE={rmse:.4f} MAPE={mape_score:.2f}%')

test_df = df.loc[test_mask, ['district', 'city', 'timestamp', 'adjusted_demand_kWh']].copy()
test_df['pred_demand'] = preds['demand']
district_mape = test_df.groupby('district').apply(
    lambda g: mape(g['adjusted_demand_kWh'], g['pred_demand'])
).sort_values(ascending=False)

print('\nDemand MAPE by district (higher is harder):')
print(district_mape)

In [ ]:
jan_2024 = df[(df['timestamp'] >= '2024-01-01') & (df['timestamp'] < '2024-02-01')].copy()
jan_2024_X = jan_2024[feature_cols]
jan_2024['pred_demand'] = models['demand'].predict(jan_2024_X)

cities_to_plot = ['Bengaluru', 'Mysuru', 'Ballari']
fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

for i, city in enumerate(cities_to_plot):
    temp = jan_2024[jan_2024['city'] == city].sort_values('timestamp')
    axes[i].plot(temp['timestamp'], temp['adjusted_demand_kWh'], label='Actual', linewidth=1.8)
    axes[i].plot(temp['timestamp'], temp['pred_demand'], label='Predicted', linewidth=1.2)
    axes[i].set_title(f'Jan 2024 Demand: {city}')
    axes[i].legend()

plt.tight_layout()
plt.show()

In [ ]:
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': models['demand'].feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 7))
sns.barplot(data=importance, x='importance', y='feature', color='#00bcd4')
plt.title('Demand Model Feature Importance (Top 20)')
plt.tight_layout()
plt.show()

residuals = y_test['demand'] - preds['demand']
plt.figure(figsize=(10, 5))
sns.histplot(residuals, bins=60, kde=True, color='#7c3aed')
plt.title('Demand Residual Distribution')
plt.xlabel('Residual')
plt.tight_layout()
plt.show()

In [ ]:
joblib.dump(models['demand'], MODEL_PATH.replace('.pkl', '_demand.pkl'))
joblib.dump(models['solar'], MODEL_PATH.replace('.pkl', '_solar.pkl'))
joblib.dump(models['surplus'], MODEL_PATH.replace('.pkl', '_surplus.pkl'))
joblib.dump(encoders, '/content/drive/MyDrive/gridwise_data/encoders.pkl')
print('Models saved to Drive')